# Amazon Product Query Assistant
End-to-end RAG system over Amazon All_Beauty products (112K products, 701K reviews)

In [ ]:
import os
os.getcwd()
os.chdir('/Users/komalpreet/Desktop/Github/Amazon_Product_Query_Assistant/')

In [ ]:
import sys
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
sys.path.append(".")

## 2. Data Loading

In [ ]:
# Load raw data from HuggingFace (first time only)
from pathlib import Path

if Path("data/raw/reviews_raw.jsonl").exists():
    print("Loading from local files...")
    reviews = [json.loads(l) for l in open("data/raw/reviews_raw.jsonl")]
    meta    = [json.loads(l) for l in open("data/raw/meta_raw.jsonl")]
    print(f"Reviews: {len(reviews):,}, Meta: {len(meta):,}")
else:
    print("Downloading from HuggingFace...")
    from datasets import load_dataset
    reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_All_Beauty", split="full", trust_remote_code=True)
    meta    = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_All_Beauty", split="full", trust_remote_code=True)
    reviews = list(reviews)
    meta    = list(meta)
    # save locally
    Path("data/raw").mkdir(parents=True, exist_ok=True)
    with open("data/raw/reviews_raw.jsonl", "w") as f:
        for row in reviews: f.write(json.dumps(dict(row)) + "\n")
    with open("data/raw/meta_raw.jsonl", "w") as f:
        for row in meta: f.write(json.dumps(dict(row)) + "\n")
    print(f"Saved. Reviews: {len(reviews):,}, Meta: {len(meta):,}")

## 3. Preprocessing

In [ ]:
# Build merged product documents (run once, loads from file after)
if Path("data/processed/products.jsonl").exists():
    print("Loading processed products...")
    with open("data/processed/products.jsonl", "r") as f:
        products = [json.loads(line) for line in f if line.strip()]
    print(f"Products loaded: {len(products):,}")
else:
    from src.preprocessor import build_products
    products = build_products(meta, reviews)
    print(f"Products built: {len(products):,}")

## 4. BM25 Retrieval

In [ ]:
# Build corpus
from src.utils import build_corpus
corpus, tokenized_corpus = build_corpus(products)

In [ ]:
# Load or build BM25 index
from src.bm25 import build_bm25, load_bm25, search_bm25

if Path("data/processed/bm25_index.pkl").exists():
    bm25, _ = load_bm25()
    print("BM25 index loaded!")
else:
    bm25 = build_bm25(tokenized_corpus)
    print("BM25 index built!")

In [ ]:
# Test BM25
results = search_bm25(bm25, products, "moisturizer for sensitive skin", top_k=5)
for r in results:
    print(f"{r['bm25_score']:.4f} | {r['title'][:70]}")

## 5. Semantic Retrieval

In [ ]:
# Load or build semantic index
from src.semantic import build_semantic_index, load_semantic_index, search_semantic
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

if Path("data/processed/faiss.index").exists():
    index, embeddings = load_semantic_index()
    print("Semantic index loaded!")
else:
    index, embeddings = build_semantic_index(corpus)
    print("Semantic index built!")

In [ ]:
# Test semantic search
results = search_semantic(index, products, "moisturizer for sensitive skin", top_k=5, model=model)
for r in results:
    print(f"{r['semantic_score']:.4f} | {r['title'][:70]}")

## 6. Hybrid Retrieval

In [ ]:
from src.hybrid import hybrid_search

results = hybrid_search(bm25, index, products, "moisturizer for sensitive skin", top_k=5, model=model)
for r in results:
    print(f"hybrid={r['hybrid_score']:.6f} | bm25={r['bm25_rank']} | sem={r['semantic_rank']} | {r['title'][:60]}")

## 7. Evaluation

In [ ]:
# Run all 10 evaluation queries
queries = [
    "moisturizer for sensitive skin",
    "vitamin c serum",
    "shampoo for curly hair",
    "something to keep my skin hydrated all day",
    "product for damaged hair",
    "gentle face wash for acne",
    "best affordable moisturizer under $20 for dry skin",
    "natural organic hair care for color treated hair",
    "anti aging cream for women over 50",
    "fragrance free products for baby sensitive skin",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")

    print("\nBM25:")
    for r in search_bm25(bm25, products, query, top_k=5):
        print(f"  {r['parent_asin']} | {r['title'][:60]}")

    print("\nSemantic:")
    for r in search_semantic(index, products, query, top_k=5, model=model):
        print(f"  {r['parent_asin']} | {r['title'][:60]}")

    print("\nHybrid:")
    for r in hybrid_search(bm25, index, products, query, top_k=5, model=model):
        print(f"  {r['parent_asin']} | {r['title'][:60]}")

In [ ]:
ground_truth = {
    "moisturizer for sensitive skin": [
        "B00YF3J4OG", "B072BKHJ7Z", "B07B8PHZPM",
        "B014T3QBLK", "B09PNWWBYX", "B01IADYIM4", "B0757RW9XK"
    ],
    "vitamin c serum": [
        "B00HSX95A8", "B07NPX9LNS", "B071177MV1",
        "B09987JTXF", "B00VQHFBBE", "B01CM8G8PI",
        "B016MNB4HQ", "B019D6WNJM", "B00S5LB08W", "B000MXRFY4"
    ],
    "shampoo for curly hair": [
        "B07L46W7J4",  # Maui Moisture Curl Quench Shampoo
        "B0884Z6NHP",  # DermOrganic Curl Shampoo Sulfate Free
        "B01M09LKNF",  # Back To Basics Curl Enhancing Shampoo
        "B01B6Z1J52",  # Curly Hair Shampoo and Conditioner Set
        "B00BUQ475W",  # Marc Anthony Strictly Curls Shampoo
        "B07TDB2M5R",  # LaCoupe Naturals for Curly Hair
    ],
    "something to keep my skin hydrated all day": [
        "B00ZM9A5NG",  # SkinResource Moisture Boost Hydragel
        "B07YLSHVXC",  # uruoi Deep Moisture Gel
        "B008JBVDUS",  # MD Formulations Moisture Defense Cream
        "B00T45V7JU",  # Curel Daily Healing Lotion
        "B00LZGWJJS",  # Ultimate Healing Moisturizing Lotion
        "B06W9L29W9",  # Hyaluronic Moisturizer Cream
    ],
    "product for damaged hair": [
        "B076J3YR85",  # AQUADERMA Hair Repairing Treatment for Damaged Hair
        "B00FN3GF6M",  # Active Organic Sea Buckthorn Oil for Damaged Hair
        "B0082Q097S",  # Avon Damage Repair 5-Day Rescue Treatment
        "B07VN6GWXP",  # Toni & Guy Damage Repair Shampoo and Conditioner
        "B013IRN9E0",  # Hair Repair Mask with Argan Oil
        "B00LCBLEGK",  # Oribe Gold Lust Transformative Masque
    ],
    "gentle face wash for acne": [
        "B001E96OGK",  # Neutrogena Oil-Free Acne Wash
        "B07T4JF9M7",  # Gentle Facial Cleanser with Rose Water
        "B019769J2W",  # Detox Handcrafted Face Wash
        "B014EUKL6O",  # Organic Face Wash By BeeFriendly
        "B0092T3OB2",  # OXY Clinical Advanced Face Wash Acne Treatment
        "B01BP5L8TA",  # 808 Dude Face Wash for Teen Acne
    ],
    "best affordable moisturizer under $20 for dry skin": [
        "B018TZTS96",  # Face Moisturizer for Dry Skin by WONDERPIEL
        "B01M1A0H4Z",  # DayTime Moisturizer for Dry Skin
        "B01IADYIM4",  # Moisturel Therapeutic Lotion
        "B00H221BSO",  # Lubrisoft Moisture Lotion for Dry Skin
        "B0757RW9XK",  # Daily Face Moisturizer for Dry Skin
    ],
    "natural organic hair care for color treated hair": [
        "B07BR5QSTK",  # Organix Sulfate Free Color Reviving
        "B01GOWKAD4",  # Argan Oil Shampoo Sulfate Free Natural
        "B0875YT39N",  # Woman to Woman Naturals Organic Hair Shampoo
        "B07TBB52VW",  # LaCoupe Naturals for Damaged Hair Sulfate Free
        "B07NH66JHX",  # Color Boost Brown Color Depositing Shampoo
    ],
    "anti aging cream for women over 50": [
        "B0109SSSYC",  # Spa Ultimate Retinol Anti Aging Night Cream
        "B01CS3OWBY",  # Allegro Anti Aging Cream
        "B01N0WY90C",  # Anti Aging Night Cream Hydrating Face Cream
        "B073WCK5H7",  # Krasa Anti-Aging Cream Kit
        "B01JZTITS6",  # Derm Essence Anti-Aging Cream
        "B00FJPMNEM",  # Anti Aging Vitamin C Cream
    ],
    "fragrance free products for baby sensitive skin": [
        "B07NXW5397",  # Huggies Natural Care Baby Wipe Fragrance Free
        "B0B7MCJ618",  # Parents Choice Fragrance Free Baby Wipes
        "B07L52XF7G",  # Simple Truth Fragrance Free Baby Wipes
        "B088RL8P6G",  # Babyology All Natural Baby Wash
        "B088RKZMLX",  # Babyology All Natural Baby Wash
        "B09HF6TGBR",  # Love Labs Organics Baby Body Butter
    ],
}

In [ ]:
import importlib
import sys

if 'src.retrieval_metrics' in sys.modules:
    del sys.modules['src.retrieval_metrics']

from src.retrieval_metrics import evaluate_all

results = evaluate_all(bm25, index, products, ground_truth, model, k=5)

for method in ["bm25", "semantic", "hybrid"]:
    print(f"\n{method.upper()} Average:")
    for metric, score in results[method]["average"].items():
        print(f"  {metric}: {score}")

## 8. RAG Pipeline

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
# OPENAI_API_KEY is loaded from the environment

In [ ]:
if 'src.tools' in sys.modules:
    del sys.modules['src.tools']

from src.tools import detect_filters, apply_tools

# test filter detection
print(detect_filters("best moisturizer under $20"))
print(detect_filters("serum rated above 4 stars"))
print(detect_filters("best affordable cream under $30 rating 4.5"))

# test apply tools on retrieved products
retrieved = hybrid_search(bm25, index, products, "moisturizer under $20", top_k=10, model=model)
filtered, filters = apply_tools(retrieved, "moisturizer under $20")
print(f"\nFilters detected: {filters}")
print(f"Before: {len(retrieved)} → After: {len(filtered)} products")
for p in filtered:
    print(f"  ${p['price']} | {p['title'][:60]}")

In [ ]:
if 'src.rag' in sys.modules:
    del sys.modules['src.rag']

from src.rag import rag_answer

result = rag_answer(
    query="best moisturizer for sensitive skin",
    bm25=bm25,
    index=index,
    products=products,
    model=model,
)
print(result["response"])

In [ ]:
if 'src.rag' in sys.modules:
    del sys.modules['src.rag']

from src.rag import rag_answer

result = rag_answer(
    query="best moisturizer for sensitive skin",
    bm25=bm25,
    index=index,
    products=products,
    model=model,
)
print(result["response"])

## 9. Guardrails

In [ ]:
if 'src.guardrails' in sys.modules:
    del sys.modules['src.guardrails']

from src.guardrails import check_input, check_output

queries = [
    "best moisturizer for sensitive skin",
    "how to make a bomb",
    "what is the weather today",
    "asdfjkl;",
    "hi",
    "best laptop under $500",
]

for q in queries:
    result = check_input(q)
    status = "✅" if result["valid"] else "❌"
    print(f"{status} '{q[:40]}' → {result['reason'] or 'valid'}")

In [ ]:
from src.rag import rag_answer

result = rag_answer(
    query="best moisturizer for sensitive skin",
    bm25=bm25,
    index=index,
    products=products,
    model=model,
)

# now test output guardrails
from src.guardrails import check_output

print(check_output(result["response"], result["retrieved"]))
print(check_output("This product cures eczema and treats acne permanently.", result["retrieved"]))
print(check_output("I don't know.", result["retrieved"]))

In [ ]:
from src.guardrails import MEDICAL_CLAIM_KEYWORDS
print(MEDICAL_CLAIM_KEYWORDS)

In [ ]:
print(check_output(
    "This product cures eczema and treats acne permanently. It has been clinically proven to heal all skin conditions and eliminate wrinkles completely within days of use.",
    result["retrieved"]
))

In [ ]:
if 'src.rag' in sys.modules:
    del sys.modules['src.rag']
from src.rag import rag_answer

# should be blocked
result = rag_answer("best laptop under $500", bm25, index, products, model)
print("Blocked:", result["response"])

# should work
result = rag_answer("best moisturizer for sensitive skin", bm25, index, products, model)
print("Valid:", result["response"][:100])

In [ ]:
if 'src.ragas_eval' in sys.modules:
    del sys.modules['src.ragas_eval']

from src.ragas_eval import run_ragas_evaluation

eval_queries = [
    "best moisturizer for sensitive skin",
    "vitamin c serum",
    "gentle face wash for acne",
]

scores = run_ragas_evaluation(eval_queries, bm25, index, products, model)